# GPT data pipeline — from token stream to `h0`

The two stages between the BPE tokenizer and the first transformer block.

| step | what | status |
|---|---|---|
| 0 | train tokenizer on `the-verdict.txt`, vocab 1000 | done -> `verdict_tokenizer.json` |
| 1 | chunking: token stream -> `(input, target)` pairs | done — 53 chunks |
| 2 | `Dataset` + `DataLoader` (batching) | done — 13 batches of 4 |
| 3 | token embedding table (`We`) | done — 1001 x 128 |
| 4 | position table (`Wp`) + add + dropout -> `h0` | done — 256 x 128 |
| 5 | wrap as a module the transformer plugs into | done — `GPTEmbedding` |

Following the GPT papers rather than any single book. The target is GPT-1 eq. (2):

$$h_0 = U W_e + W_p$$

where $U$ is the input tokens as one-hot rows, $W_e$ the token table, $W_p$ the
*learned* position table (GPT-1 explicitly rejected sinusoidal encodings).

## Step 1 — chunking

Goal: turn one flat list of ~7,000 token ids into `(input, target)` pairs where
**`target` is `input` shifted left by one position.**

No torch yet — plain Python lists, so nothing is hidden behind tensor machinery.

In [1]:
import os
import sys
from pathlib import Path

# --- data files live next to this notebook --------------------------------
HERE = Path.cwd()
assert (HERE / "the-verdict.txt").exists(), \
    f"run this notebook from the repo root (cwd is {HERE})"

# --- the BPETokenizer class lives in a companion repo ---------------------
#   https://github.com/AkshGupta2003/BPE-Tokenizer
# Set BPE_TOKENIZER_PATH to your clone, or keep it as a sibling directory.
_candidates = [Path(p) for p in [os.environ.get("BPE_TOKENIZER_PATH")] if p]
_candidates += [
    HERE.parent / "bpe-tokenizer",
    HERE.parent / "BPE-Tokenizer",
    HERE.parent / "BPT-Tokenizer",   # pre-rename clones
    HERE / "bpe-tokenizer",
]
for _c in _candidates:
    if (_c / "bpe_tokenizer.py").exists():
        sys.path.insert(0, str(_c))
        print(f"tokenizer source : {_c}")
        break
else:
    raise FileNotFoundError(
        "bpe_tokenizer.py not found. Clone the companion repo:\n"
        "  git clone git@github.com:AkshGupta2003/BPE-Tokenizer.git\n"
        "then set BPE_TOKENIZER_PATH to point at it."
    )

from bpe_tokenizer import BPETokenizer

raw_text = (HERE / "the-verdict.txt").read_text(encoding="utf-8")
tok = BPETokenizer.load(HERE / "verdict_tokenizer.json")

# vocab_size is DERIVED, never hardcoded:
#   256 base bytes + 744 learned merges = 1000  (what we asked train() for)
#                  + 1 special token (<|endoftext|> got id 1000)
#                  = 1001 distinct ids -> the embedding table needs 1001 rows
vocab_size = len(tok.vocab) + len(tok.special_tokens)

token_ids = tok.encode(raw_text)

print(f"characters  : {len(raw_text):,}")
print(f"tokens      : {len(token_ids):,}")
print(f"chars/token : {len(raw_text) / len(token_ids):.2f}")
print(f"vocab_size  : {vocab_size}")
print(f"max id seen : {max(token_ids)}  (must be < {vocab_size})")

tokenizer source : /mnt/d/Learning/bpe-tokenizer


2026-07-27 17:40:32.055 | INFO     | bpe_tokenizer:load:219 - load: 744 merges + 1 special tokens <- /mnt/d/Learning/data_loader_and_embedding_layer/verdict_tokenizer.json


characters  : 20,479
tokens      : 6,998
chars/token : 2.93
vocab_size  : 1001
max id seen : 999  (must be < 1001)


### The shift, on a tiny context

Use a context of 8 so the whole thing fits on screen. `input` and `target` are
slices of the *same* list, offset by one — the labels are free.

In [2]:
demo_ctx = 8

x = token_ids[0:demo_ctx]           # input
y = token_ids[1:demo_ctx + 1]       # same stream, shifted left by one


def piece(tid):
    """The text a single token id stands for (for display only)."""
    if tid in tok._inverse_special:
        return tok._inverse_special[tid]
    return tok.vocab[tid].decode("utf-8", errors="replace")


print("slot    input                ->  target")
for slot, (xi, yi) in enumerate(zip(x, y)):
    print(f"  {slot}   {xi:>4}  {piece(xi)!r:<12}    ->  {yi:>4}  {piece(yi)!r}")

print()
print("input  decoded:", repr(tok.decode(x)))
print("target decoded:", repr(tok.decode(y)))

slot    input                ->  target
  0     73  'I'             ->   596  ' H'
  1    596  ' H'            ->    65  'A'
  2     65  'A'             ->    68  'D'
  3     68  'D'             ->   598  ' always'
  4    598  ' always'       ->   527  ' thought'
  5    527  ' thought'      ->   441  ' Jack'
  6    441  ' Jack'         ->   399  ' Gisburn'
  7    399  ' Gisburn'      ->   663  ' rather'

input  decoded: 'I HAD always thought Jack Gisburn'
target decoded: ' HAD always thought Jack Gisburn rather'


Notice the two decoded strings are the same text, one token apart. That is the
whole trick: **one chunk of N tokens is N training examples, not 1.**

Read the ladder below — each line is a separate thing the model learns from this
single chunk. (Attention's causal mask is what lets all N happen in one forward
pass; we build that later.)

In [3]:
for i in range(1, demo_ctx + 1):
    context = token_ids[:i]
    answer = token_ids[i]
    print(f"{tok.decode(context)!r:<42} --> {piece(answer)!r}")

'I'                                        --> ' H'
'I H'                                      --> 'A'
'I HA'                                     --> 'D'
'I HAD'                                    --> ' always'
'I HAD always'                             --> ' thought'
'I HAD always thought'                     --> ' Jack'
'I HAD always thought Jack'                --> ' Gisburn'
'I HAD always thought Jack Gisburn'        --> ' rather'


### The chunking function

One subtlety worth staring at: the loop bound.

```
for i in range(0, len(ids) - context, stride):
```

The largest index the target slice touches is `i + context`. For that to exist we
need `i + context <= len(ids) - 1`, i.e. `i < len(ids) - context`. Since
`range(0, n)` stops at `n - 1`, the bound `len(ids) - context` is exactly right —
off-by-one here would either crash on the last chunk or silently drop one.

In [4]:
def make_chunks(ids, context, stride):
    """Slice a flat token stream into (input, target) pairs.

    target is input shifted left by one, so slot t of input predicts slot t of target.
    Stops early enough that the final target index (i + context) still exists.
    """
    chunks = []
    for i in range(0, len(ids) - context, stride):
        chunks.append((
            ids[i:i + context],
            ids[i + 1:i + context + 1],
        ))
    return chunks


# sanity: the tiny demo above, reproduced by the function
assert make_chunks(token_ids, demo_ctx, demo_ctx)[0] == (x, y)
print("make_chunks reproduces the demo pair:", True)

make_chunks reproduces the demo pair: True


### Real config

- **`CONTEXT = 256`** — how much history the model sees. GPT-1 used 512 and GPT-2
  used 1024, but our corpus is 6,998 tokens: at 1024 we would get *six* chunks
  total. 256 is the largest size that still leaves a usable number of chunks.

- **`STRIDE = 128`** — half the context, so consecutive chunks overlap by 128
  tokens. **This is a deliberate departure from the papers.** GPT-1 trains on
  *"contiguous sequences of 512 tokens"* — stride equal to context — because it
  had far more text than compute: re-training on text already seen would burn
  FLOPs that could have gone to fresh data.

  Our situation is inverted. 6,998 tokens, compute to spare. Halving the stride
  doubles the chunk count (27 → 53) and therefore doubles the optimizer steps per
  epoch, which makes the loss curve easier to watch.

  Be clear about what this does *not* buy, though: **overlap creates no new
  information.** The checks below show we touch the exact same 6,913 tokens either
  way. Each token simply gets visited about twice per epoch instead of once.

In [5]:
CONTEXT = 256   # how much history the model sees
STRIDE = 128    # half of CONTEXT -> consecutive chunks overlap by 128 tokens

chunks = make_chunks(token_ids, CONTEXT, STRIDE)

print(f"chunks            : {len(chunks)}")
print(f"each chunk        : input {CONTEXT} ids + target {CONTEXT} ids")
print(f"overlap           : {CONTEXT - STRIDE} tokens shared with the next chunk")
print(f"training examples : {len(chunks)} x {CONTEXT} = {len(chunks) * CONTEXT:,} next-token predictions")
print(f"batches @ size 4  : {len(chunks) // 4}")

# for reference: what the GPT setting (stride == context) would have given
print(f"\nSTRIDE == CONTEXT would give {len(make_chunks(token_ids, CONTEXT, CONTEXT))} chunks"
      f" -> {len(make_chunks(token_ids, CONTEXT, CONTEXT)) // 4} batches")

chunks            : 53
each chunk        : input 256 ids + target 256 ids
overlap           : 128 tokens shared with the next chunk
training examples : 53 x 256 = 13,568 next-token predictions
batches @ size 4  : 13

STRIDE == CONTEXT would give 27 chunks -> 6 batches


### Sanity checks

Three properties we actually want to verify rather than assume.

In [6]:
from collections import Counter

# 1. every target really is its input shifted by one  (true for any stride)
for xi, yi in chunks:
    assert yi[:-1] == xi[1:], "target is not input shifted by one"
print("1. target == input shifted left by one          ok")

# 2. consecutive chunks line up: they share exactly CONTEXT - STRIDE tokens
overlap = CONTEXT - STRIDE
for (xa, _), (xb, _) in zip(chunks, chunks[1:]):
    assert xa[STRIDE:] == xb[:overlap], "chunks do not line up"
print(f"2. consecutive chunks share exactly {overlap} tokens    ok")

# 3. coverage: how much of the corpus we touch at all
last_target_index = (len(chunks) - 1) * STRIDE + CONTEXT
touched = last_target_index + 1
print(f"3. tokens touched: {touched:,} of {len(token_ids):,} "
      f"({len(token_ids) - touched} dropped from the tail)")

# ...and the punchline: identical coverage to the no-overlap setting,
# which is what "overlap adds no new information" means concretely
no_overlap = make_chunks(token_ids, CONTEXT, CONTEXT)
assert (len(no_overlap) - 1) * CONTEXT + CONTEXT + 1 == touched
print("   identical coverage to STRIDE == CONTEXT -> no new data, just re-visits")

# 4. how many chunks each token shows up in, as an input
seen = Counter(t for k in range(len(chunks))
               for t in range(k * STRIDE, k * STRIDE + CONTEXT))
print(f"4. appearances per token: {dict(sorted(Counter(seen.values()).items()))}")
print(f"   (sum = {sum(seen.values()):,} = {len(chunks)} chunks x {CONTEXT} slots)")

1. target == input shifted left by one          ok
2. consecutive chunks share exactly 128 tokens    ok
3. tokens touched: 6,913 of 6,998 (85 dropped from the tail)
   identical coverage to STRIDE == CONTEXT -> no new data, just re-visits
4. appearances per token: {1: 256, 2: 6656}
   (sum = 13,568 = 53 chunks x 256 slots)


The dropped tail is the remainder that cannot fill a whole chunk. GPT drops it
too — with 40 GB of text a partial chunk is not worth the special-casing. Ours is
85 tokens (~1.2% of the corpus).

**What the overlap actually cost us.** Check 4 prints `{1: 256, 2: 6656}`: the
first 128 and last 128 tokens of the covered range appear in only one chunk, while
everything between them appears in two. So the middle of the story receives twice
the gradient of its edges. That lopsidedness is the real price of
`STRIDE < CONTEXT` — not wasted compute, which we have plenty of.

**Position slots, on the other hand, are unaffected.** Every chunk is a full
`CONTEXT` tokens long, so every position slot `0..255` appears in exactly
`len(chunks)` examples — slot 0 and slot 255 are trained equally, whatever the
stride. That matters in step 4: no row of `W_p` ends up starved relative to
another.

## Step 3 — the token embedding table

Turn integer ids into vectors the model can do arithmetic with.

Token id 412 is just a *label*. It is not bigger or better than 411 — the ids fell
out of BPE merge order, which is arbitrary. Feeding raw ids into a network would
imply an ordering that does not exist.

So: build a lookup table with one row per vocabulary entry, and let training decide
what the rows contain.

This is the `U W_e` half of GPT-1 eq. (2):

$$h_0 = U W_e + W_p$$

In [7]:
import torch

torch.manual_seed(123)   # reproducible random init

# Step 2 (Dataset/DataLoader) is not built yet, so stack a batch by hand.
# This is exactly what a DataLoader will hand us later: two [B, T] int tensors.
BATCH = 4

inputs = torch.tensor([x for x, _ in chunks[:BATCH]], dtype=torch.long)
targets = torch.tensor([y for _, y in chunks[:BATCH]], dtype=torch.long)

print(f"inputs  : {tuple(inputs.shape)}  dtype={inputs.dtype}")
print(f"targets : {tuple(targets.shape)}  dtype={targets.dtype}")

# dtype MUST be an integer type - nn.Embedding indexes with these, not multiplies
print(f"\nrow 0, first 10 ids : {inputs[0, :10].tolist()}")
print(f"decoded             : {tok.decode(inputs[0, :10].tolist())!r}")

inputs  : (4, 256)  dtype=torch.int64
targets : (4, 256)  dtype=torch.int64

row 0, first 10 ids : [73, 596, 65, 68, 598, 527, 441, 399, 663, 258]
decoded             : 'I HAD always thought Jack Gisburn rather a'


### Why the paper writes it as a multiplication

`U W_e` looks like a matrix multiply, but `U` holds one-hot rows — so the multiply
merely *selects* rows. Worth proving to yourself once: the cell below does the same
job both ways and checks the answers match.

In [8]:
# tiny world: 5 tokens, 3 dimensions
tiny_vocab, tiny_dim = 5, 3
tiny_ids = torch.tensor([2, 0, 4])

W_e = torch.randn(tiny_vocab, tiny_dim)

# --- way 1: the paper's notation. build U as one-hot rows, then matmul ---
U = torch.nn.functional.one_hot(tiny_ids, num_classes=tiny_vocab).float()
by_matmul = U @ W_e

# --- way 2: what code actually does. index the rows ---
by_lookup = W_e[tiny_ids]

print("U (one-hot rows, ids 2/0/4):")
print(U.int())
print(f"\nU @ W_e  ->\n{by_matmul}")
print(f"\nW_e[ids] ->\n{by_lookup}")
print(f"\nexactly equal : {torch.equal(by_matmul, by_lookup)}")

# why nobody builds U for real: its size is batch x context x vocab
real = BATCH * CONTEXT * vocab_size
gpt2 = 512 * 1024 * 50257
print(f"\nU for our batch : {real:,} floats  ({real * 4 / 1e6:.1f} MB)")
print(f"U for GPT-2      : {gpt2:,} floats  ({gpt2 * 4 / 1e9:.0f} GB)  <- not happening")

U (one-hot rows, ids 2/0/4):
tensor([[0, 0, 1, 0, 0],
        [1, 0, 0, 0, 0],
        [0, 0, 0, 0, 1]], dtype=torch.int32)



U @ W_e  ->
tensor([[-0.9724, -0.7550,  0.3239],
        [-0.1115,  0.1204, -0.3696],
        [ 0.2350,  0.6653,  0.3528]])

W_e[ids] ->
tensor([[-0.9724, -0.7550,  0.3239],
        [-0.1115,  0.1204, -0.3696],
        [ 0.2350,  0.6653,  0.3528]])

exactly equal : True

U for our batch : 1,025,024 floats  (4.1 MB)
U for GPT-2      : 26,349,142,016 floats  (105 GB)  <- not happening


### The real table

`EMB_DIM` is ours to choose. GPT-1 and GPT-2-small both used 768; GPT-3 went up to
12288. We use **128**, for two reasons:

- our corpus is 6,998 tokens. At 768 dims the token table alone would be ~769k
  parameters, to model a 20 KB short story — pure memorisation, and slow on CPU.
- it keeps shapes readable. `[4, 256, 128]` is unambiguous, whereas `[4, 256, 256]`
  hides which axis is which when `CONTEXT` is also 256.

It is just a knob — nothing below depends on the value.

In [9]:
EMB_DIM = 128

token_embedding = torch.nn.Embedding(vocab_size, EMB_DIM)

print(f"table shape : {tuple(token_embedding.weight.shape)}   (vocab_size x EMB_DIM)")
print(f"parameters  : {token_embedding.weight.numel():,}")
print(f"trainable   : {token_embedding.weight.requires_grad}")

token_emb = token_embedding(inputs)

print(f"\ninputs     {tuple(inputs.shape)}        ints   (token ids)")
print(f"token_emb  {tuple(token_emb.shape)}   floats  <- [B, T, EMB_DIM]")

table shape : (1001, 128)   (vocab_size x EMB_DIM)
parameters  : 128,128
trainable   : True

inputs     (4, 256)        ints   (token ids)
token_emb  (4, 256, 128)   floats  <- [B, T, EMB_DIM]


The table starts as **random noise** — `nn.Embedding` initialises from N(0, 1).
Nothing in it is meaningful yet; training is what gives the rows meaning. (GPT-2
actually uses N(0, 0.02); we will match that in step 5.)

Recall from step 0 that only **731 of our 1001** ids appear in the corpus. The other
270 rows will never receive a gradient — they stay random forever. That is the price
of byte-level BPE: you carry all 256 byte rows so that nothing is ever an unknown
token. Check 3 below demonstrates this directly.

In [10]:
from collections import Counter

# 1. embedding really is just a row lookup
row_id = inputs[0, 0].item()
assert torch.equal(token_emb[0, 0], token_embedding.weight[row_id])
print(f"1. token_emb[0,0] == weight[{row_id}]                    ok")

# 2. the SAME token id anywhere gets the SAME vector
flat = inputs.flatten().tolist()
repeated = next(tid for tid, n in Counter(flat).most_common() if n > 1)
where = (inputs == repeated).nonzero()
(r1, s1), (r2, s2) = where[0].tolist(), where[1].tolist()
same = torch.equal(token_emb[r1, s1], token_emb[r2, s2])
print(f"2. token {repeated} ({piece(repeated)!r}) sits at [{r1},{s1}] and [{r2},{s2}]")
print(f"   their vectors are identical: {same}    <-- THE PROBLEM")

# 3. only rows present in the batch receive gradient
token_embedding.zero_grad(set_to_none=True)
token_embedding(inputs).sum().backward()
touched = (token_embedding.weight.grad.abs().sum(dim=1) > 0).sum().item()
print(f"3. rows given gradient by this batch : {touched} of {vocab_size}")
print(f"   distinct ids in the batch         : {len(set(flat))}")
token_embedding.zero_grad(set_to_none=True)   # leave state clean

1. token_emb[0,0] == weight[73]                    ok
2. token 44 (',') sits at [0,36] and [0,44]
   their vectors are identical: True    <-- THE PROBLEM


3. rows given gradient by this batch : 313 of 1001
   distinct ids in the batch         : 313


**Check 2 is the entire motivation for step 4.** The table has no idea *where* a
token sat: `' the'` at slot 3 and `' the'` at slot 200 come out as the same vector.

So right now `"dog bites man"` and `"man bites dog"` would produce the same *set* of
vectors. Attention — which we build later — is also order-agnostic, so nothing
downstream can recover the difference. The order information has to be injected
here or it is gone for good.

Step 4 fixes it with a second table keyed on **slot** instead of **token**.

## Step 2 — `Dataset` and `DataLoader`

`make_chunks` built all 53 pairs up front. A `Dataset` instead stores the flat stream
**once** and slices it on demand — the version that still works on a 40 GB corpus.
Slicing a torch tensor returns a *view*, so the 53 chunks cost no extra memory.

`DataLoader` then handles batching, shuffling, and dropping the ragged last batch.

In [11]:
from torch.utils.data import Dataset, DataLoader


class GPTDataset(Dataset):
    """Slices a flat token stream into (input, target) pairs on demand."""

    def __init__(self, ids, context, stride):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.context = context
        self.starts = range(0, len(ids) - context, stride)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, k):
        i = self.starts[k]
        return self.ids[i:i + self.context], self.ids[i + 1:i + self.context + 1]


dataset = GPTDataset(token_ids, CONTEXT, STRIDE)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,     # shuffles chunk ORDER; tokens inside a chunk stay put
    drop_last=True,   # 53 chunks / 4 leaves a remainder -> dropped
    num_workers=0,    # tiny dataset on CPU: workers would only add overhead
)

print(f"dataset : {len(dataset)} chunks")
print(f"loader  : {len(loader)} batches of 4 "
      f"({len(dataset) - len(loader) * 4} chunk dropped by drop_last)")

dataset : 53 chunks
loader  : 13 batches of 4 (1 chunk dropped by drop_last)


In [12]:
# 1. the Dataset must agree with step 1's make_chunks
for k in (0, 1, len(dataset) - 1):
    x_, y_ = dataset[k]
    assert x_.tolist() == chunks[k][0] and y_.tolist() == chunks[k][1]
print("1. Dataset agrees with make_chunks             ok")

# 2. slices are views into one tensor, not copies
x0, _ = dataset[0]
print(f"2. slice shares storage with the stream: {x0.data_ptr() == dataset.ids.data_ptr()}")

# 3. what the loader actually hands us
bx, by = next(iter(loader))
print(f"3. batch: inputs {tuple(bx.shape)}, targets {tuple(by.shape)}, dtype {bx.dtype}")
print(f"   target still shifted by one: {torch.equal(by[0, :-1], bx[0, 1:])}")

1. Dataset agrees with make_chunks             ok
2. slice shares storage with the stream: True


3. batch: inputs (4, 256), targets (4, 256), dtype torch.int64
   target still shifted by one: True


## Step 4 — position embeddings, and `h0`

Step 3 left us with a problem: `','` at slot 36 and `','` at slot 44 produced
identical vectors. Fix it with a second table looked up by **slot number** instead of
token id, then add the two together.

GPT-1 used *learned* position embeddings — an ordinary trainable table — explicitly
rejecting the sinusoidal encodings of the original Transformer paper.

In [13]:
pos_embedding = torch.nn.Embedding(CONTEXT, EMB_DIM)

positions = torch.arange(CONTEXT)        # 0, 1, 2, ... 255
pos_emb = pos_embedding(positions)

print(f"table shape : {tuple(pos_embedding.weight.shape)}    (CONTEXT x EMB_DIM)")
print(f"parameters  : {pos_embedding.weight.numel():,}")
print(f"\npos_emb   {tuple(pos_emb.shape)}       one row per slot")
print(f"token_emb {tuple(token_emb.shape)}  one row per token in the batch")

# [B, T, D] + [T, D] -> broadcasting aligns the trailing dims and reuses the same
# position rows for every sequence in the batch. That is exactly right: slot 3 is
# slot 3 regardless of which chunk it came from.
h0_nodrop = token_emb + pos_emb
print(f"h0        {tuple(h0_nodrop.shape)}  <- token + position")

table shape : (256, 128)    (CONTEXT x EMB_DIM)
parameters  : 32,768

pos_emb   (256, 128)       one row per slot
token_emb (4, 256, 128)  one row per token in the batch
h0        (4, 256, 128)  <- token + position


In [14]:
# THE STEP-3 PROBLEM, RE-CHECKED
v1, v2 = h0_nodrop[r1, s1], h0_nodrop[r2, s2]
print(f"token {repeated} ({piece(repeated)!r}) at [{r1},{s1}] and [{r2},{s2}]")
print(f"  token_emb identical : {torch.equal(token_emb[r1, s1], token_emb[r2, s2])}")
print(f"  h0 identical        : {torch.equal(v1, v2)}   <-- FIXED")
print(f"  now differ by       : {(v1 - v2).abs().max():.4f}  (max abs)")

# GPT-1 applies dropout 0.1 to the embedding sum
dropout = torch.nn.Dropout(0.1)

dropout.train()
frac_train = (dropout(h0_nodrop) == 0).float().mean()
dropout.eval()
frac_eval = (dropout(h0_nodrop) == 0).float().mean()

print(f"\ndropout p=0.1   train mode : {frac_train:.1%} of entries zeroed")
print(f"                eval  mode : {frac_eval:.1%}   (a no-op at eval)")

token 44 (',') at [0,36] and [0,44]
  token_emb identical : True
  h0 identical        : False   <-- FIXED
  now differ by       : 4.3072  (max abs)



dropout p=0.1   train mode : 9.9% of entries zeroed
                eval  mode : 0.0%   (a no-op at eval)


Two things that make the add work:

- **Broadcasting.** `token_emb` is `[4, 256, 128]` and `pos_emb` is `[256, 128]`.
  Torch lines up the trailing dimensions and reuses the same position rows for all 4
  sequences — correct, because slot 3 means the same thing in every chunk.
- **Adding, not concatenating.** Both tables are `EMB_DIM` wide, so the sum stays
  `EMB_DIM` wide. Concatenating would double the width at every layer for no gain;
  the model learns to keep the two kinds of information separable on its own.

## Step 5 — wrap it as a module

Everything above in the form the transformer will use. Two tables, an add, dropout.

Two details worth keeping:

- **init std 0.02.** `nn.Embedding` defaults to N(0, 1), but GPT-2 uses N(0, 0.02).
  At N(0, 1) the embedding values would dwarf everything downstream.
- **the context assert.** `Wp` has exactly `context` rows, so a longer sequence has
  nowhere to look. This is precisely why GPT-2 cannot accept more than 1024 tokens.

In [15]:
class GPTEmbedding(torch.nn.Module):
    """h0 = We[tokens] + Wp[positions], then dropout.   GPT-1 eq. (2)."""

    def __init__(self, vocab_size, context, emb_dim, dropout=0.1):
        super().__init__()
        self.context = context
        self.tok_emb = torch.nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = torch.nn.Embedding(context, emb_dim)
        self.drop = torch.nn.Dropout(dropout)

        # GPT-2 init: N(0, 0.02), not nn.Embedding's default N(0, 1)
        torch.nn.init.normal_(self.tok_emb.weight, std=0.02)
        torch.nn.init.normal_(self.pos_emb.weight, std=0.02)

    def forward(self, idx):
        _, T = idx.shape
        assert T <= self.context, f"sequence length {T} > context {self.context}"
        pos = torch.arange(T, device=idx.device)
        return self.drop(self.tok_emb(idx) + self.pos_emb(pos))

In [16]:
torch.manual_seed(123)
emb = GPTEmbedding(vocab_size, CONTEXT, EMB_DIM, dropout=0.1)
emb.eval()                       # dropout off -> deterministic output

h0 = emb(inputs)

tok_p = emb.tok_emb.weight.numel()
pos_p = emb.pos_emb.weight.numel()

print(emb)
print(f"\ninputs {tuple(inputs.shape)} -> h0 {tuple(h0.shape)}")
print(f"\ntoken table    {str(tuple(emb.tok_emb.weight.shape)):<12} {tok_p:>7,}")
print(f"position table {str(tuple(emb.pos_emb.weight.shape)):<12} {pos_p:>7,}")
print(f"total                       {tok_p + pos_p:>8,} parameters")
print(f"\nweight std after init: {emb.tok_emb.weight.std():.4f}  (target 0.02)")

# the context guard
try:
    emb(torch.zeros(1, CONTEXT + 1, dtype=torch.long))
except AssertionError as e:
    print(f"context guard works: {e}")

GPTEmbedding(
  (tok_emb): Embedding(1001, 128)
  (pos_emb): Embedding(256, 128)
  (drop): Dropout(p=0.1, inplace=False)
)

inputs (4, 256) -> h0 (4, 256, 128)

token table    (1001, 128)  128,128
position table (256, 128)    32,768
total                        160,896 parameters

weight std after init: 0.0200  (target 0.02)
context guard works: sequence length 257 > context 256


In [17]:
import math

# GPT-1 predicts with softmax(h_n @ W_e.T) -- the SAME token table, transposed.
# The transformer blocks will go between h0 and h_n; weight tying closes the loop.
logits = h0 @ emb.tok_emb.weight.T

print(f"h0      {tuple(h0.shape)}")
print(f"We.T    {tuple(emb.tok_emb.weight.T.shape)}")
print(f"logits  {tuple(logits.shape)}   <- one score per vocab entry, per slot")

# and it already lines up with what a loss function needs
loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), targets.flatten())
print(f"\nuntrained loss : {loss:.4f}")
print(f"random guessing: ln({vocab_size}) = {math.log(vocab_size):.4f}")
print("matching those two is the standard check that nothing is wired up wrong.")

h0      (4, 256, 128)
We.T    (128, 1001)
logits  (4, 256, 1001)   <- one score per vocab entry, per slot



untrained loss : 6.9083
random guessing: ln(1001) = 6.9088
matching those two is the standard check that nothing is wired up wrong.


## Done — what `h0` is

`[4, 256, 128]`: for each of 4 sequences, for each of 256 slots, a 128-number vector
encoding *what token is here* **and** *where here is*. That is the tensor the first
transformer block consumes.

The untrained loss landing on `ln(vocab_size)` is the standard smoke test: an
untrained model should be exactly as good as guessing uniformly. If it came out much
lower, something would be leaking targets into inputs; much higher, and the init or
the shapes would be wrong.

**What comes next** (a different repo): the transformer blocks — causal self-attention
plus an MLP, pre-LayerNorm, residual connections — stacked `n_layer` times, then
`softmax(h_n · We^T)` reusing the token table we built here.